# Mini Project 1 — Analysis Notebook

**Your name:**  Auli Badoni
**Dataset:**  Billboard Top 100
**Date:**  6th May 2026

This notebook has four sections. Work through them in order. Each section has instructions and a code cell to fill in. Add markdown cells to explain your thinking as you go — that writing is part of the assignment.

When you're done, publish this notebook to your GitHub repository and submit the URL to Canvas.

In [1]:
# Setup — run this cell first
# If any package is missing, it will install automatically
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for pkg in ["pandas", "plotly", "kaleido", "nbformat"]:
    try:
        __import__(pkg)
    except ImportError:
        print(f"Installing {pkg}...")
        install(pkg)

import pandas as pd
import plotly.express as px

print("Setup complete.")

Setup complete.


In [2]:
# Additional imports needed for this analysis
import numpy as np
import plotly.graph_objects as go
import plotly.io as pio
import requests
from PIL import Image
from io import BytesIO
print("All libraries loaded successfully")

All libraries loaded successfully


---

## Section 1 — Overview
# Billboard Hot 100 Analysis | 2023–2026

**Dataset:** Weekly Billboard Hot 100 chart data collected using the `billboard.py` 
Python package, which scrapes chart data from Billboard.com. Covers 174 weekly 
charts from January 7, 2023 to May 2, 2026 — 17,400 rows, 100 songs per week.

**Why this dataset:** Billboard chart data offers a structured way to study how 
audiences collectively respond to music over time, connecting to HCD themes of 
attention, popularity, and sustained engagement with creative products.

**Three analytical questions:**

1. Which songs and artists had the strongest chart presence between 2023 and 2026, 
   based on total weekly appearances, Top 10 appearances, and 1 placements?
2. How do songs typically move through the Billboard Hot 100: do they slowly climb, 
   debut high, remain stable, or drop quickly after peaking?
3. Has chart behavior changed across 2023–2026 in terms of new entries, average 
   weeks on chart, and rank movement volatility?

**What a practitioner would do with these findings:** Music label A&R teams and 
streaming platform curators could use these trajectory patterns to identify 
slow-burn hits early and adjust playlist promotion timing accordingly.

**Data provenance note:** Data was scraped via `billboard.py` from Billboard.com. 
Gaps may exist where Billboard's site structure changed or rate limits were hit. 
Top-ranked songs (ranks 1–4) frequently have missing image URLs due to scraping 
behavior — handled in cleaning.

---

## Section 2 — Data Profile

**Shape:** 17,400 rows × 16 columns (plus 2 derived columns added during cleaning)

**What each column represents:**

- `chart_date` — the Saturday the chart was published (weekly)
- `rank` — song's position that week (1 = best, 100 = worst)
- `song` — song title
- `artist` — full credited artist string including features
- `last_week` — rank from the previous week (null for new entries & re-entries)
- `peak_rank` — best rank the song has ever achieved (cumulative, set by Billboard)
- `weeks_on_board` — total weeks the song has appeared on the chart (cumulative)
- `is_new` — True if this is the song's first week on the chart
- `image_url` — Billboard CDN URL for the song's cover art (frequently null for top ranks)
- `year` — extracted from chart_date
- `month` — extracted from chart_date
- `decade` — decade of chart_date (all 2020s in this dataset)
- `is_top_10` — True if rank ≤ 10
- `is_number_one` — True if rank == 1
- `rank_change` — how many positions the song moved vs last week (positive = up)
- `movement_type` — categorical: new, up, down, same, re-entry

**Derived columns added during cleaning:**
- `primary_artist` — first-billed artist extracted from the full artist string
- `featured_artists` — list of all non-primary credited artists

**Data quality issues found:**
- `last_week` and `rank_change` are null for 2,788 rows — expected behavior 
  for new entries (2,044) and re-entries (744), not treated as missing data
- `image_url` is null for 696 rows — Billboard does not consistently serve 
  images for top-ranked songs via scraping
- `artist` column contains inconsistent collaboration formatting 
  (& vs Featuring vs X vs ,) — resolved via primary_artist extraction

**Columns this analysis focuses on:**
- `rank`, `chart_date`, `primary_artist`, `song` — core identifiers and metrics
- `is_top_10`, `is_number_one` — for Q1 dominance analysis
- `rank_change`, `movement_type` — for Q2 trajectory and Q3 volatility
- `is_new`, `weeks_on_board` — for Q3 trend analysis over time

In [3]:
# Load your dataset
df = pd.read_csv('week 6/data/billboard_hot100_2023_2026.csv')  # ← replace with your filename

print(df.shape)
df.head()

(17400, 16)


,chart_date,rank,song,artist,last_week,peak_rank,weeks_on_board,is_new,image_url,year,month,decade,is_top_10,is_number_one,rank_change,movement_type
0,2023-01-07,1,All I Want For Christmas Is You,Mariah Carey,1.0,1,58,False,NaN,2023,1,2020,True,True,0.0,same
1,2023-01-07,2,Rockin' Around The Christmas Tree,Brenda Lee,2.0,2,52,False,NaN,2023,1,2020,True,False,0.0,same
2,2023-01-07,3,Jingle Bell Rock,Bobby Helms,3.0,3,49,False,NaN,2023,1,2020,True,False,0.0,same
3,2023-01-07,4,Last Christmas,Wham!,5.0,4,31,False,NaN,2023,1,2020,True,False,1.0,up
4,2023-01-07,5,A Holly Jolly Christmas,Burl Ives,4.0,4,32,False,https://charts-static.billboard.com/img/1998/0...,2023,1,2020,True,False,-1.0,down


In [4]:
# Check column types and missing values
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 17400 entries, 0 to 17399
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   chart_date      17400 non-null  str    
 1   rank            17400 non-null  int64  
 2   song            17400 non-null  str    
 3   artist          17400 non-null  str    
 4   last_week       14612 non-null  float64
 5   peak_rank       17400 non-null  int64  
 6   weeks_on_board  17400 non-null  int64  
 7   is_new          17400 non-null  bool   
 8   image_url       16704 non-null  str    
 9   year            17400 non-null  int64  
 10  month           17400 non-null  int64  
 11  decade          17400 non-null  int64  
 12  is_top_10       17400 non-null  bool   
 13  is_number_one   17400 non-null  bool   
 14  rank_change     14612 non-null  float64
 15  movement_type   17400 non-null  str    
dtypes: bool(3), float64(2), int64(6), str(5)
memory usage: 1.8 MB


In [5]:
# Summary statistics for numeric columns
df.describe()

,rank,last_week,peak_rank,weeks_on_board,year,month,decade,rank_change
count,17400.0000,14612.000000,17400.000000,17400.000000,17400.000000,17400.000000,17400.0,14612.000000
mean,50.5000,45.918765,32.615000,13.720575,2024.206897,6.143678,2020.0,-2.272310
std,28.8669,27.456324,27.359857,13.656719,0.984248,3.488511,0.0,12.507993
min,1.0000,1.000000,1.000000,1.000000,2023.000000,1.000000,2020.0,-73.000000
25%,25.7500,22.000000,8.000000,4.000000,2023.000000,3.000000,2020.0,-7.000000
50%,50.5000,45.000000,26.000000,10.000000,2024.000000,6.000000,2020.0,-1.000000
75%,75.2500,69.000000,53.000000,19.000000,2025.000000,9.000000,2020.0,3.000000
max,100.0000,100.000000,100.000000,112.000000,2026.000000,12.000000,2020.0,79.000000


In [6]:
# Plotly/Kaleido: fetch Chromium for static image export (pipe y to the prompt)
!echo y | plotly_get_chrome



Plotly will install a copy of Google Chrome to be used for generating static images of plots.
Chrome will be installed at: None
Do you want to proceed? [y/n] Installing Chrome for Plotly...
Chrome installed successfully.
The Chrome executable is now located at: /Users/auli/Library/Application Support/choreographer/deps/chrome-mac-arm64/Google Chrome for Testing.app/Contents/MacOS/Google Chrome for Testing


**My data profile notes:**

When I first loaded the dataset I noticed it was largely clean out of the box — 
no corrupt rows, no duplicate chart weeks, and all core fields like rank, song, 
and artist were fully populated. The 2,788 null values in last_week and 
rank_change looked alarming at first but turned out to be expected — they all 
correspond to new entries and re-entries, which by definition have no previous 
week to reference.

The artist column immediately stood out as inconsistent — the same collaboration 
can appear as "Drake & 21 Savage", "Drake Featuring 21 Savage", or 
"Metro Boomin, The Weeknd & 21 Savage" depending on the song. This will need 
cleaning before any artist-level aggregation.

The weeks_on_board column is cumulative — it reflects a song's total chart 
history, not just weeks in this dataset. This means some songs like Mariah 
Carey's Christmas songs appear with 50+ weeks even though they only charted 
for a few weeks within my date range. I will need to handle this carefully 
for Q1 and Q3.

One question this raises: how do I count an artist's chart presence fairly 
when one artist releases 30 songs at once (Morgan Wallen) vs another who 
has 3 songs that each dominate for months (Taylor Swift)?

### Fix Data Types
Convert chart_date to datetime and ensure numeric columns are correct type.

In [7]:
### Fix Data Types

# Convert chart_date from string to datetime
df['chart_date'] = pd.to_datetime(df['chart_date'])

# Convert numeric columns (coerce errors handles nulls gracefully)
numeric_cols = ['rank', 'last_week', 'peak_rank', 'weeks_on_board', 'rank_change']
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Confirm types
print(df.dtypes)

chart_date        datetime64[us]
rank                       int64
song                         str
artist                       str
last_week                float64
peak_rank                  int64
weeks_on_board             int64
is_new                      bool
image_url                    str
year                       int64
month                      int64
decade                     int64
is_top_10                   bool
is_number_one               bool
rank_change              float64
movement_type                str
dtype: object


### Handle Missing Values
Inspect nulls — note that last_week and rank_change being null 
is expected for new entries and re-entries.

In [8]:
# Check missing values
print("Missing values per column:")
print(df.isnull().sum())
print()

# Confirm nulls are only from new entries and re-entries
print("Null last_week breakdown by movement_type:")
print(df[df['last_week'].isna()]['movement_type'].value_counts())
print()

# Drop only rows where critical fields are null
before = df.shape[0]
df = df.dropna(subset=['rank', 'song', 'artist'])
after = df.shape[0]
print(f"Rows before: {before} | Rows after: {after} | Dropped: {before - after}")

Missing values per column:
chart_date           0
rank                 0
song                 0
artist               0
last_week         2788
peak_rank            0
weeks_on_board       0
is_new               0
image_url          696
year                 0
month                0
decade               0
is_top_10            0
is_number_one        0
rank_change       2788
movement_type        0
dtype: int64

Null last_week breakdown by movement_type:
movement_type
new         2044
re-entry     744
Name: count, dtype: int64

Rows before: 17400 | Rows after: 17400 | Dropped: 0


###  Extract Primary Artist and Featured Artists
Many entries include collaborations — extract the first-billed 
artist as primary_artist for clean grouping.


In [9]:
import re

def extract_primary_artist(artist):
    primary = re.split(r' & | Featuring | X | x |, ', artist)[0]
    return primary.strip()

def extract_featured_artists(artist):
    parts = re.split(r' & | Featuring | X | x |, ', artist)
    features = [p.strip() for p in parts[1:] if p.strip()]
    return features if features else None

df['primary_artist'] = df['artist'].apply(extract_primary_artist)
df['featured_artists'] = df['artist'].apply(extract_featured_artists)

# Create unique song key
df['song_key'] = df['song'] + ' — ' + df['primary_artist']

# Sanity check
print("Sample with features:")
print(df[df['featured_artists'].notna()][['artist', 'primary_artist', 'featured_artists']].head(5).to_string())
print(f"\nNew shape: {df.shape}")

Sample with features:
                                                                     artist                      primary_artist                     featured_artists
9                                                    Sam Smith & Kim Petras                           Sam Smith                         [Kim Petras]
15  Bing Crosby With Ken Darby Singers & John Scott Trotter & His Orchestra  Bing Crosby With Ken Darby Singers  [John Scott Trotter, His Orchestra]
18                                                David Guetta & Bebe Rexha                        David Guetta                         [Bebe Rexha]
19              Frank Sinatra With The Orchestra & Chorus Of Gordon Jenkins    Frank Sinatra With The Orchestra           [Chorus Of Gordon Jenkins]
20                                                        Drake & 21 Savage                               Drake                          [21 Savage]

New shape: (17400, 19)


###  Build Image Lookup + Final Sanity Check (Pulling Images for the graphs)
Build an artist and song image lookup dictionary for use in charts,
then confirm the dataset is clean and ready for analysis.

In [18]:
# --- IMAGE LOOKUP: song -> image_url ---
image_lookup = (df[df['image_url'].notna()]
                .groupby('song_key')['image_url']
                .first()
                .to_dict())

# --- IMAGE LOOKUP: artist -> image_url ---
artist_image_lookup = (df[df['image_url'].notna()]
                       .sort_values('chart_date', ascending=False)
                       .drop_duplicates(subset='primary_artist')
                       .set_index('primary_artist')['image_url']
                       .to_dict())

print(f"Songs with image URLs: {len(image_lookup)}")
print(f"Artists with image URLs: {len(artist_image_lookup)}")
print()

# --- FINAL SANITY CHECK ---
print("=== FINAL DATASET SUMMARY ===")
print(f"Total rows: {df.shape[0]}")
print(f"Total columns: {df.shape[1]}")
print(f"Date range: {df['chart_date'].min().date()} → {df['chart_date'].max().date()}")
print(f"Unique songs: {df['song'].nunique()}")
print(f"Unique primary artists: {df['primary_artist'].nunique()}")
print(f"Unique chart weeks: {df['chart_date'].nunique()}")
print()
print("Movement type breakdown:")
print(df['movement_type'].value_counts())
print()
print("Top 5 songs by weeks on chart:")
print(df.groupby('song_key')['weeks_on_board'].max()
      .sort_values(ascending=False).head(5))

Songs with image URLs: 2228
Artists with image URLs: 506

=== FINAL DATASET SUMMARY ===
Total rows: 17400
Total columns: 19
Date range: 2023-01-07 → 2026-05-02
Unique songs: 2181
Unique primary artists: 506
Unique chart weeks: 174

Movement type breakdown:
movement_type
down        7882
up          5493
new         2044
same        1237
re-entry     744
Name: count, dtype: int64

Top 5 songs by weeks on chart:
song_key
Lose Control — Teddy Swims                        112
Beautiful Things — Benson Boone                    89
All I Want For Christmas Is You — Mariah Carey     79
A Bar Song (Tipsy) — Shaboozey                     77
Wildflower — Billie Eilish                         72
Name: weeks_on_board, dtype: int64


---

## Section 3 — Analysis

Answer your three research questions using pandas. Each question should have:

1. A markdown cell stating the question
2. A code cell with the analysis
3. A markdown cell with your interpretation — what does the result mean?

You may need to clean or reshape the data before you can answer a question. That's normal — document what you did and why.

**Question 1:** *Which songs and artists had the strongest chart presence between 2023 and 2026, 
   based on total weekly appearances, Top 10 appearances, and 1 placements?*

In [31]:
## Section 4 — Analysis
### Chart 1: Longest-Running Top 10 Songs (2023–2026)

# --- TOP 10 MOST CHARTED SONGS: Overall + Per Year ---

# --- BUILD OVERALL TOP 10 ---
overall = df.groupby(['song', 'primary_artist']).agg(
    top10_weeks   = ('is_top_10', 'sum'),
    number1_weeks = ('is_number_one', 'sum'),
    total_weeks   = ('rank', 'count'),
    peak_rank     = ('rank', 'min')
).reset_index()

overall = overall.sort_values('top10_weeks', ascending=True).tail(10)
overall['label'] = '<b>' + overall['song'] + '</b><br>  ' + overall['primary_artist']
overall['hover'] = (
    '<b>' + overall['song'] + '</b><br>' +
    overall['primary_artist'] + '<br>' +
    'Peak Rank: #' + overall['peak_rank'].astype(str) + '<br>' +
    'Weeks in Top 10: ' + overall['top10_weeks'].astype(str) + '<br>' +
    'Weeks at #1: ' + overall['number1_weeks'].astype(str)
)

# --- PLOT ---
fig = go.Figure()

fig.add_trace(go.Bar(
    y=overall['label'],
    x=overall['top10_weeks'],
    orientation='h',
    name='Weeks in Top 10',
    marker_color='#7b2d8b',
    hovertext=overall['hover'],
    hoverinfo='text'
))

fig.add_trace(go.Bar(
    y=overall['label'],
    x=overall['number1_weeks'],
    orientation='h',
    name='Weeks at #1',
    marker_color='#e91e8c',
    hovertext=overall['hover'],
    hoverinfo='text'
))

fig.update_layout(
    title=dict(
        text='<b>Billboard Hot 100</b><br>'
             '<sup>Top 10 Most Charted Songs (2023–2026)</sup>',
        font=dict(size=20, color='#2d1a2e'),
        x=0.5
    ),
    barmode='overlay',
    xaxis=dict(
        title='Weeks in Top 10',
        gridcolor='#e8c8e0',
        tickfont=dict(color='#666666')
    ),
    yaxis=dict(
        tickfont=dict(size=11, color='#2d1a2e'),
        ticklabelposition='outside',
        ticksuffix='  '
    ),
    plot_bgcolor='#fdf0f5',
    paper_bgcolor='#f5eaf5',
    legend=dict(
        orientation='h',
        yanchor='top',
        y=-0.12,
        xanchor='center',
        x=0.5,
        font=dict(size=12, color='#2d1a2e'),
        bgcolor='rgba(253,240,245,0.8)',
        bordercolor='#e8c8e0',
        borderwidth=1
    ),

    height=600,
        margin=dict(l=200, r=100, t=100, b=60)

)

fig.show()
#fig.write_image('chart1_songs_top10.png', width=1400, height=700, scale=2)
#print("Saved as chart1_songs_top10.png")

**Interpretation:**  
*(What does this result tell you? Is it what you expected? What would you want to investigate further?)*

**Question 2:** *How do songs typically move through the Billboard Hot 100: do they slowly climb, 
   debut high, remain stable, or drop quickly after peaking?*

In [ ]:
# Your analysis for Question 2


In [ ]:
# Q2 — STEP 1: classify each song's trajectory shape on the Hot 100
# Inputs (already present in df_top20):
#   song_key, chart_date, rank, week_number, chart_score, song, primary_artist
# Convention used in df_top20: chart_score = 101 - rank (higher = better),
# week_number = cumulative count per song starting at 1, sorted by chart_date ascending.

def _trajectory_for(group):
    peak_idx      = group['chart_score'].idxmax()
    peak_week     = group.loc[peak_idx, 'week_number']
    total_weeks   = group['week_number'].max()
    peak_position = peak_week / total_weeks

    if peak_position <= 0.25:
        return 'Debut High'
    elif peak_position >= 0.60:
        return 'Slow Climber'
    return 'Spike & Drop'


trajectory_by_song = (
    df_top20
    .groupby('song_key', sort=False)
    .apply(_trajectory_for)
)
df_top20['trajectory'] = df_top20['song_key'].map(trajectory_by_song)

print(df_top20['trajectory'].value_counts())
df_top20[['song_key', 'song', 'primary_artist', 'trajectory']].drop_duplicates(subset='song_key')

**Interpretation:**  
*(What does this result tell you?)*

**Question 3:** *Has chart behavior changed across 2023–2026 in terms of new entries, average 
   weeks on chart, and rank movement volatility?*

In [ ]:
# Your analysis for Question 3


**Interpretation:**  
*(What does this result tell you?)*

---

## Section 4 — Visualization

Create at least one visualization that supports one of your analysis findings. Your chart should:

- Have a title that states the finding, not just the data (e.g., "Satisfaction scores drop sharply after age 40" not "Satisfaction by age")
- Have labeled axes
- Use a chart type appropriate for your data (bar for categorical comparison, scatter for relationships, line for trends over time)

Below the chart, explain in a markdown cell: why you chose this chart type, and what you want the reader to take away from it.

In [ ]:
# Your visualization
# Example structure — replace with your actual columns and finding

# fig = px.bar(
#     df,
#     x="your_category_column",
#     y="your_value_column",
#     title="Your finding stated as a claim",
#     labels={"your_category_column": "Readable label", "your_value_column": "Readable label"}
# )
# fig.show()


**Chart rationale:**  
*(Why this chart type? What should the reader take away?)*

---

## Section 5 — Conclusions

Write 3–5 sentences summarizing what you found. Address these questions:

- What is the most important thing your analysis revealed?
- What surprised you?
- What would you investigate next if you had more time or data?
- What are the limitations of this analysis — what can't you conclude from this data?

Then complete the competency claim below.

**Summary of findings:**  
*(Write your 3–5 sentence conclusion here.)*

---

## Competency Claim

In a `mp1.md` file in your GitHub repository, write a short competency claim (2–4 sentences) for each domain you feel this project demonstrates. Be specific — cite something you actually did in this notebook.

Domains covered by this project typically include:
- **C3 — Data cleaning and file handling** (if you cleaned or reshaped data)
- **C5 — Data analysis with pandas** (answering questions with code)
- **C6 — Data visualization** (your chart)
- **C7 — Critical evaluation and professional judgment** (your interpretation and limitations section)

You don't have to claim every domain — only the ones your work actually demonstrates.